# Incremental docking of prepared inputs

Public `dock_many(...)` executes prepared problems one at a time and returns
`DockingOutcome` records. See [the contract](incremental_docking.md) and
[issue #36](https://github.com/uibcdf/dockingmt/issues/36).

This software control reuses exact captured PDBQT from the retained minimal Vina
result. No automatic preparation occurs. The tiny overlapping geometry and
declared control state labels do not establish docking quality or chemical
qualification. The recorded preparation remains unassessed.
Select the Python 3.14 `molsyssuite@uibcdf_3.14` kernel.

In [1]:
import base64
import json
import sys
from collections import Counter
from copy import deepcopy
from pathlib import Path
from tempfile import TemporaryDirectory

import dockingmt as dmt
import pyunitwizard as puw

assert sys.version_info[:2] == (3, 14)
assert Path(sys.prefix).name == 'molsyssuite@uibcdf_3.14'
root = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / 'devguide/validation/data/audit/minimal_vina_result.json').is_file()
)
saved = json.loads(
    (root / 'devguide/validation/data/audit/minimal_vina_result.json').read_text()
)
artifacts = saved['provenance']['backend_artifacts']
sizes = dmt.verify_captured_inputs(artifacts)
inputs = {
    role: base64.b64decode(artifact['content_base64']).decode()
    for role, artifact in artifacts.items()
}
box = saved['provenance']['backend_box']
domain = dmt.BoxRegion(
    puw.quantity(box['center'], box['unit']),
    puw.quantity(box['size'], box['unit']),
)
protocol = dmt.VinaProtocol(
    seed=123, cpu=1, exhaustiveness=1, n_poses=2,
    capture_backend_inputs=True,
)

def prepared_problem(label, unsupported=False):
    return dmt.DockingProblem(
        inputs['receptor'], inputs['partner'], domain,
        constraints=[{'kind': 'unsupported-control'}] if unsupported else [],
        metadata={
            'ligand_id': label,
            'partner_state_id': label + '-state',
            'receptor_state_id': 'receptor-control',
        },
    )

consumed = []

def source():
    for label, unsupported in [('first', False), ('bad', True), ('third', False)]:
        consumed.append(label)
        yield prepared_problem(label, unsupported)

stream = dmt.dock_many(source(), protocol, on_error='record')
assert consumed == []
print('Python:', sys.version.split()[0], 'environment:', Path(sys.prefix).name)
print('Verified input byte sizes:', sizes)
print('No input consumed before advancing the iterator.')

Python: 3.14.7 environment: molsyssuite@uibcdf_3.14
Verified input byte sizes: {'receptor': 317, 'partner': 181}
No input consumed before advancing the iterator.


## Incremental persistence and explicit item failures

Write each outcome as one JSON line. The consumer keeps counters and one
comparison snapshot, rather than collecting all results. The middle problem
requests an unsupported constraint; its capability failure is recorded and
the following prepared problem still executes.

In [2]:
counts = Counter()
first_scientific_record = None
with TemporaryDirectory(prefix='dockingmt-batch-validation-') as temporary:
    destination = Path(temporary) / 'outcomes.jsonl'
    with destination.open('w') as output:
        for outcome in stream:
            counts[outcome.status] += 1
            snapshot = outcome.to_dict()
            output.write(json.dumps(snapshot, allow_nan=False) + '\n')
            assert consumed[-1] == outcome.problem_info['metadata']['ligand_id']
            if outcome.status == 'success':
                assert dmt.audit_result(snapshot['result'])['status'] == 'consistent'
                if first_scientific_record is None:
                    first_scientific_record = deepcopy(snapshot['result'])
            else:
                assert outcome.error['code'] == 'DMT-E003'
            print(json.dumps({
                'index': outcome.index,
                'status': outcome.status,
                'ligand_id': outcome.problem_info['metadata']['ligand_id'],
                'failure_code': outcome.error['code'] if outcome.error else None,
            }))
    restored_counts = Counter()
    with destination.open() as stored:
        for line in stored:
            record = json.loads(line)
            restored = dmt.DockingOutcome.from_dict(record)
            assert restored.to_dict() == record
            restored_counts[restored.status] += 1
    assert counts == restored_counts == {'success': 2, 'failure': 1}
assert consumed == ['first', 'bad', 'third']
print('Independent JSON Lines roundtrips:', dict(restored_counts))

{"index": 0, "status": "success", "ligand_id": "first", "failure_code": null}
{"index": 1, "status": "failure", "ligand_id": "bad", "failure_code": "DMT-E003"}


{"index": 2, "status": "success", "ligand_id": "third", "failure_code": null}
Independent JSON Lines roundtrips: {'success': 2, 'failure': 1}


## Native equivalence and early close

Compare one streamed result with a standalone seeded call, excluding only
wall-clock elapsed time. Closing a new iterator before execution consumes no
source item. Explicit seeds are passed unchanged; no affinity-map reuse,
global ranking, campaign identities or scientific comparability are inferred.

In [3]:
individual = dmt.dock(prepared_problem('first'), protocol).to_dict()
first_scientific_record['provenance'].pop('elapsed_seconds')
individual['provenance'].pop('elapsed_seconds')
assert first_scientific_record == individual
assert 'molsysviewer' not in sys.modules
consumed.clear()
unstarted = dmt.dock_many(source(), protocol)
unstarted.close()
assert consumed == []
print('Scores, geometry, identity, hashes, descriptors and provenance match dock().')
print('Early close consumed no input; no viewer was loaded.')

Scores, geometry, identity, hashes, descriptors and provenance match dock().
Early close consumed no input; no viewer was loaded.
